# Phase 1 Analysis: Palestine/Gaza Humanitarian Impact Dataset

This notebook loads and summarizes the Phase 1 CIVID dataset covering verified incidents from Gaza and the West Bank based on OCHA and humanitarian agency reports.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the Phase 1 data
events_df = pd.read_csv('../data/phase1_palestine/events.csv')
persons_df = pd.read_csv('../data/phase1_palestine/persons.csv')
sources_df = pd.read_csv('../data/phase1_palestine/sources.csv')

print(f"Loaded {len(events_df)} events")
print(f"Loaded {len(persons_df)} person records")
print(f"Loaded {len(sources_df)} verified sources")

## Dataset Overview

In [ ]:
# Display basic statistics
print("=" * 80)
print("PHASE 1 DATASET SUMMARY")
print("=" * 80)
print(f"\nTotal Events: {len(events_df)}")
print(f"Total Person Records: {len(persons_df)}")
print(f"Verified Sources: {len(sources_df)}")

# Fatality and injury counts
total_fatalities = events_df['fatalities'].sum()
total_injuries = events_df['injuries'].sum()
events_with_casualties = (events_df['fatalities'].notna() & (events_df['fatalities'] > 0)).sum()

print(f"\nTOTAL DOCUMENTED CASUALTIES:")
print(f"  Fatalities: {int(total_fatalities) if pd.notna(total_fatalities) else 'N/A'}")
print(f"  Injuries: {int(total_injuries) if pd.notna(total_injuries) else 'N/A'}")
print(f"  Events with Fatalities: {events_with_casualties}")

## Event Categories and Impact Types

In [ ]:
# Events by location type
print("\nEVENT DISTRIBUTION BY LOCATION TYPE:")
print("-" * 50)
location_counts = events_df['location_type'].value_counts()
for loc_type, count in location_counts.items():
    print(f"  {loc_type}: {count}")

# Events by verification status
print("\nEVENT VERIFICATION STATUS:")
print("-" * 50)
verification_counts = events_df['verification_status'].value_counts()
for status, count in verification_counts.items():
    pct = (count / len(events_df)) * 100
    print(f"  {status}: {count} ({pct:.1f}%)")

# Events by confidence level
print("\nEVENT CONFIDENCE BREAKDOWN:")
print("-" * 50)
confidence_counts = events_df['confidence_level'].value_counts()
for conf, count in confidence_counts.items():
    pct = (count / len(events_df)) * 100
    print(f"  {conf}: {count} ({pct:.1f}%)")

## Timeline of Events

In [ ]:
# Convert event_date to datetime for timeline analysis
events_df['event_date'] = pd.to_datetime(events_df['event_date'])

# Sort by date
events_timeline = events_df.sort_values('event_date')

print("\nEVENT TIMELINE:")
print("-" * 80)
print(f"Date Range: {events_timeline['event_date'].min().date()} to {events_timeline['event_date'].max().date()}")
print(f"\nEvents by Date (earliest to latest):")
print("-" * 80)

for idx, row in events_timeline.head(10).iterrows():
    print(f"\n{row['event_date'].date()} | {row['event_id']:8} | {row['location_type']:15} | {row['location'][:40]}")
    if pd.notna(row['fatalities']) and row['fatalities'] > 0:
        print(f"           Fatalities: {int(row['fatalities'])}", end="")
    if pd.notna(row['injuries']) and row['injuries'] > 0:
        print(f" | Injuries: {int(row['injuries'])}", end="")
    print()

## Fatalities by Event Type

In [ ]:
# Fatalities by location type (impact category)
events_with_fatalities = events_df[events_df['fatalities'].notna() & (events_df['fatalities'] > 0)]

print("\nFATALITIES BY EVENT CATEGORY:")
print("-" * 50)

fatalities_by_type = events_with_fatalities.groupby('location_type')['fatalities'].agg(['sum', 'count', 'mean'])
fatalities_by_type.columns = ['Total', 'Events', 'Avg per Event']
fatalities_by_type = fatalities_by_type.sort_values('Total', ascending=False)

for cat, row in fatalities_by_type.iterrows():
    print(f"  {cat:20} | Total: {int(row['Total']):3} | Events: {int(row['Events']):2} | Avg: {row['Avg per Event']:.1f}")

print(f"\n{'TOTAL FATALITIES':20} | {int(fatalities_by_type['Total'].sum()):3}")

## Source Reliability Assessment

In [ ]:
print("\nSOURCE RELIABILITY SCORES:")
print("-" * 80)
print(f"{'Source ID':30} | {'Type':20} | {'Reliability':12}")
print("-" * 80)

for idx, row in sources_df.iterrows():
    source_events = len(events_df[events_df['source_id'] == row['source_id']])
    print(f"{row['source_id']:30} | {row['source_type']:20} | {row['reliability_score']:.2f} ({source_events:2} events)")

## Events Table

In [ ]:
# Display events table with key columns
display_cols = ['event_id', 'event_date', 'location', 'location_type', 'fatalities', 'injuries', 'verification_status', 'confidence_level']
events_display = events_df[display_cols].copy()
events_display['event_date'] = events_display['event_date'].dt.strftime('%Y-%m-%d')
events_display = events_display.sort_values('event_date')

print("\n" + "=" * 120)
print("PHASE 1 EVENTS")
print("=" * 120)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)
display(events_display)

## Person Records Table

In [ ]:
# Display person records with key columns
person_display_cols = ['record_id', 'event_id', 'victim_name', 'victim_age', 'victim_gender', 'victim_role', 'child_flag', 'civilian_flag', 'verification_status']
persons_display = persons_df[person_display_cols].copy()

print("\n" + "=" * 120)
print("PHASE 1 PERSON RECORDS")
print("=" * 120)
display(persons_display)

## Data Quality Notes

- All events are verified through humanitarian agency reports (OCHA, UNRWA, OHCHR)
- Confidence levels reflect source reliability and corroboration
- Person records are populated only when sources explicitly provide individual-level data
- Missing values indicate data not available in source material
- All records include source citations for full traceability